# UGRF v1 — AUTO Benchmark Reproduction

This notebook reproduces the **AUTO** result from the frozen UGRF Paper-1 benchmark using the public `ugrf` Python package.

The canonical AUTO specification is:

- **Dataset:** AUTO intermittent-demand panel
- **Panel shape:** 3,000 SKUs × 24 periods
- **Forecast horizon:** 6 periods
- **Historical utility origin:** period 12
- **Final forecast origin:** period 18
- **Loss:** six-period cumulative absolute error
- **TSB parameters:** `alpha = 0.10`, `beta = 0.10`
- **UGRF gate:** use Renewal only when historical utility is strictly positive
- **Frozen UGRF / TSB error ratio:** **0.9761486035**

The notebook checks that the installed package reproduces that frozen result within the paper's numerical tolerance.

> This is a **reproduction notebook**, not a tuning notebook. Do not change the model parameters and then interpret the result as the published benchmark.


## 1. Install and import UGRF

If UGRF is already installed, leave the installation command commented out.


In [ ]:
# Uncomment in a fresh environment:
# %pip install -U ugrf


In [ ]:
from pathlib import Path
import json
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ugrf
from ugrf import UtilityGatedRenewal

print("UGRF version:", ugrf.__version__)


## 2. Download the public AUTO benchmark

The AUTO panel used by the frozen experiment is the public intermittent-demand dataset hosted in the GluonTS intermittent-datasets repository.

The download is cached locally as `data/auto.json`, so rerunning the notebook does not download it again.


In [ ]:
AUTO_URL = (
    "https://raw.githubusercontent.com/canerturkmen/gluon-ts/"
    "intermittent-datasets/datasets/intermittent_auto/test/data.json"
)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
AUTO_PATH = DATA_DIR / "auto.json"

if not AUTO_PATH.exists():
    print("Downloading AUTO benchmark...")
    req = urllib.request.Request(
        AUTO_URL,
        headers={"User-Agent": "ugrf-reproduction-notebook/0.1.0"},
    )
    with urllib.request.urlopen(req) as response:
        AUTO_PATH.write_bytes(response.read())

print("Dataset path:", AUTO_PATH.resolve())


## 3. Load the AUTO panel

The source can be either a JSON array or newline-delimited JSON. The loader below handles both forms.


In [ ]:
def load_gluonts_json(path):
    text = Path(path).read_text(encoding="utf-8").strip()

    try:
        payload = json.loads(text)
        if isinstance(payload, dict):
            payload = [payload]
    except json.JSONDecodeError:
        payload = [
            json.loads(line)
            for line in text.splitlines()
            if line.strip()
        ]

    rows = [np.asarray(record["target"], dtype=float) for record in payload]
    panel = np.vstack(rows)

    panel[~np.isfinite(panel)] = 0.0
    panel[panel < 0] = 0.0

    return panel


panel = load_gluonts_json(AUTO_PATH)

print("AUTO shape:", panel.shape)
print("Overall zero rate:", round(float(np.mean(panel == 0)), 4))
print("Mean demand:", round(float(panel.mean()), 4))
print("Mean positive demand:", round(float(panel[panel > 0].mean()), 4))

assert panel.shape == (3000, 24), (
    f"Expected AUTO shape (3000, 24), got {panel.shape}"
)


## 4. Inspect several AUTO demand series

AUTO is an intermittent-demand panel, so many periods are zero and non-zero demand arrives irregularly.


In [ ]:
plt.figure(figsize=(11, 5))

for sku in range(5):
    plt.plot(
        np.arange(panel.shape[1]),
        panel[sku],
        marker="o",
        linewidth=1,
        label=f"SKU {sku}",
    )

plt.xlabel("Period")
plt.ylabel("Demand")
plt.title("AUTO benchmark — example demand histories")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Freeze the canonical AUTO protocol

The benchmark uses exactly one historical utility origin for AUTO:

\[
\mathcal{O} = \{12\}
\]

At origin 12, both TSB and Renewal are trained only on periods `0:12` and evaluated on the next six periods.

The final forecast is then generated at origin 18 and evaluated on periods 18–23.

This gives:

```text
0 -------- 11 | 12 -------- 17 | 18 -------- 23
  training      utility window    final test
```

The utility result determines which expert receives forecast responsibility at the final origin.


In [ ]:
HORIZON = 6
UTILITY_ORIGINS = (12,)
FINAL_ORIGIN = 18

EXPECTED_UGRF_TSB_RATIO = 0.9761486034991342
FREEZE_ATOL = 5e-6

print("Horizon:", HORIZON)
print("Utility origins:", UTILITY_ORIGINS)
print("Final origin:", FINAL_ORIGIN)
print("Frozen AUTO UGRF/TSB ratio:", EXPECTED_UGRF_TSB_RATIO)


## 6. Fit the frozen UGRF model

UGRF consists of two experts:

1. **TSB**, the fallback intermittent-demand baseline.
2. **Renewal**, the pooled age-dependent occurrence model with partially pooled positive-demand magnitude.

Historical forecast utility determines which expert is used for each SKU.


In [ ]:
model = UtilityGatedRenewal(horizon=HORIZON)

model.fit(
    panel,
    utility_origins=UTILITY_ORIGINS,
    final_origin=FINAL_ORIGIN,
)

ugrf_forecast = model.predict_cumulative()

print("Number of final forecasts:", len(ugrf_forecast))
print("Renewal selected:", int(model.renewal_active_.sum()))
print("TSB selected:", int((~model.renewal_active_).sum()))
print(
    "Renewal activation share:",
    round(float(model.renewal_active_.mean()), 4),
)


## 7. Score the final six-period benchmark window

The published comparison uses **total six-period cumulative absolute error** across the valid SKUs.

For each SKU:

\[
e_i(M) =
\left|
D^{(6)}_{i,18}
-
\widehat{D}^{(6),M}_{i,18}
\right|
\]

and the dataset-level ratio is:

\[
R_{\text{UGRF/TSB}}
=
\frac{\sum_i e_i(\text{UGRF})}
     {\sum_i e_i(\text{TSB})}.
\]


In [ ]:
scores = model.score_final_window()

results = pd.DataFrame({
    "sku": np.arange(panel.shape[0]),
    "actual_cumulative": scores["actual"],
    "forecast_UGRF": scores["forecast"],
    "forecast_TSB": model.tsb_forecast_,
    "forecast_Renewal": model.renewal_forecast_,
    "error_UGRF": scores["absolute_error"],
    "error_TSB": scores["tsb_absolute_error"],
    "error_Renewal": scores["renewal_absolute_error"],
    "ugrf_utility": model.utility_,
    "selected_model": model.selected_model_,
    "renewal_active": model.renewal_active_.astype(int),
    "utility_evidence_count": model.utility_evidence_count_,
})

results.head(10)


In [ ]:
total_ugrf = float(np.nansum(results["error_UGRF"]))
total_tsb = float(np.nansum(results["error_TSB"]))
total_renewal = float(np.nansum(results["error_Renewal"]))

ratio_ugrf_tsb = total_ugrf / total_tsb
ratio_renewal_tsb = total_renewal / total_tsb

summary = pd.DataFrame({
    "model": ["UGRF", "TSB", "Renewal"],
    "total_absolute_error": [
        total_ugrf,
        total_tsb,
        total_renewal,
    ],
    "ratio_to_TSB": [
        ratio_ugrf_tsb,
        1.0,
        ratio_renewal_tsb,
    ],
    "mean_absolute_error": [
        np.nanmean(results["error_UGRF"]),
        np.nanmean(results["error_TSB"]),
        np.nanmean(results["error_Renewal"]),
    ],
    "median_absolute_error": [
        np.nanmedian(results["error_UGRF"]),
        np.nanmedian(results["error_TSB"]),
        np.nanmedian(results["error_Renewal"]),
    ],
})

summary


## 8. Frozen reproduction guard

The original research code requires AUTO to reproduce:

```text
UGRF / TSB = 0.9761486034991342
```

within an absolute tolerance of `5e-6`.

The cell below deliberately fails if the installed package no longer reproduces the frozen benchmark.


In [ ]:
difference = ratio_ugrf_tsb - EXPECTED_UGRF_TSB_RATIO

print(f"Reproduced UGRF/TSB ratio : {ratio_ugrf_tsb:.12f}")
print(f"Frozen target             : {EXPECTED_UGRF_TSB_RATIO:.12f}")
print(f"Difference                : {difference:.12e}")
print(f"Allowed absolute tolerance: {FREEZE_ATOL:.1e}")

assert np.isclose(
    ratio_ugrf_tsb,
    EXPECTED_UGRF_TSB_RATIO,
    rtol=0.0,
    atol=FREEZE_ATOL,
), (
    "Frozen AUTO reproduction FAILED. "
    f"Got {ratio_ugrf_tsb:.12f}; "
    f"expected {EXPECTED_UGRF_TSB_RATIO:.12f}."
)

print("\n✓ Frozen AUTO reproduction PASSED")


A ratio of approximately **0.97615** corresponds to roughly **2.39% lower aggregate absolute error than the fixed-parameter TSB reference** on the frozen AUTO benchmark:

\[
1 - 0.9761486 \approx 0.02385.
\]

This is a dataset-level aggregate comparison, not a claim that UGRF is better for every individual SKU.


In [ ]:
improvement_pct = 100.0 * (1.0 - ratio_ugrf_tsb)
print(f"UGRF aggregate error reduction vs TSB: {improvement_pct:.3f}%")


## 9. Visualize benchmark performance


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(summary["model"], summary["ratio_to_TSB"])
plt.axhline(1.0, linewidth=1)
plt.ylabel("Total absolute error / TSB error")
plt.title("AUTO benchmark — error ratio to TSB")
plt.tight_layout()
plt.show()


## 10. Examine the utility gate

The frozen gate is:

\[
w_i = \mathbb{I}(U_i > 0)
\]

where:

\[
U_i =
\sum_o
\left[
L_{i,o}(\text{TSB})
-
L_{i,o}(\text{Renewal})
\right].
\]

For AUTO there is only one utility origin, so the sign of that historical six-period loss difference determines the final expert.


In [ ]:
gate_summary = (
    results.groupby("selected_model", as_index=False)
    .agg(
        n_skus=("sku", "count"),
        share=("sku", lambda s: len(s) / len(results)),
        mean_utility=("ugrf_utility", "mean"),
        median_utility=("ugrf_utility", "median"),
        mean_final_ugrf_error=("error_UGRF", "mean"),
    )
)

gate_summary


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(results["ugrf_utility"], bins=50)
plt.axvline(0.0, linewidth=1)
plt.xlabel("Historical utility: TSB loss − Renewal loss")
plt.ylabel("Number of SKUs")
plt.title("AUTO — distribution of historical UGRF utility")
plt.tight_layout()
plt.show()


## 11. Gate diagnostics

The final test window lets us examine whether the historical gate selected Renewal on SKUs where Renewal subsequently beat TSB.

These are **diagnostics only**. The final outcome is never used to construct the gate.


In [ ]:
valid = (
    np.isfinite(results["error_TSB"])
    & np.isfinite(results["error_Renewal"])
)

diag = results.loc[valid].copy()

diag["renewal_final_wins"] = (
    diag["error_Renewal"] < diag["error_TSB"]
)

active = diag["renewal_active"].astype(bool)
renewal_wins = diag["renewal_final_wins"].astype(bool)

tp = int(np.sum(active & renewal_wins))
fp = int(np.sum(active & ~renewal_wins))
fn = int(np.sum(~active & renewal_wins))
tn = int(np.sum(~active & ~renewal_wins))

gate_diagnostics = pd.Series({
    "n_skus": len(diag),
    "activation_share": float(active.mean()),
    "true_positive": tp,
    "false_positive": fp,
    "false_negative": fn,
    "true_negative": tn,
    "activation_precision": tp / max(tp + fp, 1),
    "renewal_opportunity_recall": tp / max(tp + fn, 1),
    "actual_renewal_win_share": float(renewal_wins.mean()),
})

gate_diagnostics


## 12. Inspect a high-utility SKU

This example selects the SKU with the highest historical Renewal utility and compares the two final expert forecasts with realized six-period cumulative demand.


In [ ]:
sku_id = int(
    results.sort_values("ugrf_utility", ascending=False)
    .iloc[0]["sku"]
)

row = results.loc[results["sku"] == sku_id].iloc[0]

pd.Series({
    "SKU": sku_id,
    "selected_model": row["selected_model"],
    "historical_utility": row["ugrf_utility"],
    "actual_cumulative": row["actual_cumulative"],
    "TSB_forecast": row["forecast_TSB"],
    "Renewal_forecast": row["forecast_Renewal"],
    "UGRF_forecast": row["forecast_UGRF"],
    "TSB_abs_error": row["error_TSB"],
    "Renewal_abs_error": row["error_Renewal"],
    "UGRF_abs_error": row["error_UGRF"],
})


In [ ]:
history_periods = np.arange(FINAL_ORIGIN)
future_periods = np.arange(FINAL_ORIGIN, FINAL_ORIGIN + HORIZON)

plt.figure(figsize=(11, 4))

plt.plot(
    history_periods,
    panel[sku_id, :FINAL_ORIGIN],
    marker="o",
    label="Observed history",
)

plt.plot(
    future_periods,
    panel[sku_id, FINAL_ORIGIN:FINAL_ORIGIN + HORIZON],
    marker="o",
    label="Final held-out demand",
)

plt.axvline(FINAL_ORIGIN - 0.5, linewidth=1)

plt.xlabel("Period")
plt.ylabel("Demand")
plt.title(
    f"AUTO SKU {sku_id} — selected expert: "
    f"{row['selected_model']}"
)
plt.legend()
plt.tight_layout()
plt.show()


## 13. Renewal hazard diagnostic for the selected SKU

UGRF separates **structural evidence** from **forecast responsibility**. The Renewal hazard can therefore be inspected without using it as a reason to override the utility gate.


In [ ]:
from ugrf.diagnostics import (
    hazard_curve,
    occurrence_probabilities,
    series_summary,
)

history = panel[sku_id, :FINAL_ORIGIN]

diagnostic = series_summary(
    model.renewal_model_,
    history,
    horizon=HORIZON,
)

pd.Series(diagnostic, name="value")


In [ ]:
ages, hazards = hazard_curve(
    model.renewal_model_,
    history,
    max_age=18,
)

plt.figure(figsize=(8, 4))
plt.plot(ages, hazards, marker="o")
plt.xlabel("Demand age")
plt.ylabel("Conditional occurrence probability")
plt.title(f"AUTO SKU {sku_id} — fitted Renewal hazard")
plt.tight_layout()
plt.show()


In [ ]:
future_occurrence = occurrence_probabilities(
    model.renewal_model_,
    history,
    horizon=HORIZON,
)

pd.DataFrame({
    "forecast_lead": np.arange(1, HORIZON + 1),
    "marginal_occurrence_probability": future_occurrence,
})


## 14. Find where UGRF helped most — and where it hurt

This is useful for understanding the allocation problem rather than looking only at one aggregate number.


In [ ]:
results["ugrf_gain_vs_tsb"] = (
    results["error_TSB"] - results["error_UGRF"]
)

columns = [
    "sku",
    "selected_model",
    "ugrf_utility",
    "actual_cumulative",
    "forecast_TSB",
    "forecast_Renewal",
    "forecast_UGRF",
    "error_TSB",
    "error_UGRF",
    "ugrf_gain_vs_tsb",
]

print("Largest UGRF gains versus TSB")
display(
    results.sort_values(
        "ugrf_gain_vs_tsb",
        ascending=False,
    )[columns].head(10)
)

print("\nLargest UGRF losses versus TSB")
display(
    results.sort_values(
        "ugrf_gain_vs_tsb",
        ascending=True,
    )[columns].head(10)
)


## 15. Save the SKU-level reproduction output

Saving this table makes it easy to investigate particular SKUs or compare package versions later.


In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

output_path = OUTPUT_DIR / "AUTO_ugrf_v1_reproduction.csv"
results.to_csv(output_path, index=False)

print("Saved:", output_path.resolve())


# Interpretation

A successful run should reproduce the frozen AUTO ratio:

```text
UGRF / TSB ≈ 0.9761486035
```

The important methodological point is not that Renewal should always replace TSB. The Renewal candidate is allowed to forecast an SKU only when it previously demonstrated lower rolling-origin loss than TSB.

In other words:

\[
\text{mechanism evidence} \neq \text{forecast utility}.
\]

The AUTO result is one component of the frozen three-dataset benchmark. The other canonical ratios are:

| Dataset | Frozen UGRF / TSB |
|---|---:|
| Car Parts | 0.9918971759 |
| AUTO | **0.9761486035** |
| RAF | 0.8905672730 |

For the full research reproduction, use the benchmark scripts included in the UGRF GitHub repository.


## Next: UGRF v1.1

UGRF v1.0 is a point-forecast model. A natural next extension is a **temporally calibrated conformal prediction layer** that leaves the frozen point forecast and gate unchanged while providing uncertainty intervals around cumulative demand forecasts.
